# EcoSort — Entrenar el modelo (Colab)

Entrena el clasificador de residuos con **transfer learning** (ADR 0001) y lo exporta a
**TFLite** para la Raspberry Pi. Este notebook es un envoltorio delgado: clona el repo y
llama a los scripts de `vision/` — el entrenamiento en sí vive ahí, versionado y con CI,
no acá adentro.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)*.
Después: *Entorno de ejecución → Ejecutar todas*.

Al final se descarga un `.zip` con:
- `model_int8.tflite` → el que va a la Pi
- `model_fp32.tflite` → referencia para comparar velocidad
- `labels.txt` → nombres de las clases en el orden del modelo
- `final.keras` → modelo completo, para seguir afinando más adelante

## 1. Verificar GPU

In [ ]:
!nvidia-smi -L || print("SIN GPU: activala en Entorno de ejecución -> Cambiar tipo de entorno")
import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

## 2. Clonar el repo (dataset + scripts de `vision/`)

No se instala `vision/requirements.txt`: Colab ya trae TensorFlow, scikit-learn, numpy y
matplotlib preinstalados, y forzar las versiones pineadas del repo puede romper el soporte
de GPU que ya viene configurado acá.

In [ ]:
!git clone --depth 1 -q https://github.com/tpII/2026-g5-ecosort.git
%cd 2026-g5-ecosort/vision

## 3. Entrenar

`--backbone mobilenetv2` es el que está desplegado hoy en `raspberry/modelo/`. Para
comparar contra `mobilenetv3small` en igualdad de condiciones, correr esta celda dos
veces cambiando solo el backbone (mismo dataset, mismo split, mismas épocas) y comparar
`runs/<backbone>/history.json` y el tamaño del `.tflite` exportado.

In [ ]:
!python train.py --backbone mobilenetv2 --head-epochs 15 --finetune-epochs 15

## 4. Evaluar en test (imágenes que el modelo nunca vio)

Matriz de confusión + precision/recall/F1 por clase — el entregable que pide la cátedra.

In [ ]:
!python evaluate.py --run-dir runs/mobilenetv2
from IPython.display import Image, display
display(Image("runs/mobilenetv2/confusion_matrix.png"))

## 5. Exportar a TFLite (int8 + fp32 de referencia)

`--io-dtype float32` (default) es el que ya corre en la Pi — ver `raspberry/inferencia_pi.py`.
El fp32 es solo para comparar latencia, no se despliega.

In [ ]:
!python export_tflite.py --run-dir runs/mobilenetv2

## 6. Armar el paquete para la Pi

Mismos nombres que ya usa `raspberry/modelo/` — no hace falta renombrar nada al copiarlos.

In [ ]:
import json, shutil
from pathlib import Path

run_dir = Path("runs/mobilenetv2")
paquete = Path("ecosort_modelo")
paquete.mkdir(exist_ok=True)

classes = json.loads((run_dir / "classes.json").read_text())
(paquete / "labels.txt").write_text("\n".join(classes) + "\n")

# mismos nombres que espera raspberry/modelo/ (ver guia-instalacion-raspberry.md)
shutil.copy(run_dir / "model_int8.tflite", paquete / "ecosort_int8.tflite")
shutil.copy(run_dir / "model_fp32.tflite", paquete / "ecosort_fp32.tflite")
shutil.copy(run_dir / "final.keras", paquete / "ecosort_mobilenetv2.keras")

for f in sorted(paquete.iterdir()):
    print(f"{f.name:28s} {f.stat().st_size / 1e6:.1f} MB")

shutil.make_archive("ecosort_modelo", "zip", paquete)

## 7. Descargar

Copiar `ecosort_int8.tflite`, `ecosort_fp32.tflite` y `labels.txt` del `.zip` a
`raspberry/modelo/`, siguiendo `docs/guia-instalacion-raspberry.md`.
`ecosort_mobilenetv2.keras` se puede guardar aparte (no va a la Pi) por si hace falta
seguir afinando el modelo más adelante.

In [ ]:
from google.colab import files
files.download("ecosort_modelo.zip")